## Surrogate vs. Thermochemical library
### *Time evolution analysis*

Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from silx.io.dictdump import dicttoh5, h5todict
from scipy import ndimage
from reading import read_dataset
import plotting as pltt

Define inputs

In [ ]:
models = ['surrogate', 'thermolib']
scheme = 'strang'
nb_nodes = 1401
nb_iters = 2200
qoi = {
    'names': ['Y_O2', 'Th', 'Tv'],
    'labels': {
        'Y_O2': r"$Y_{\text{O}_2}$",
        'Y_O': r"$Y_{\text{O}}$",
        'Th': r"$T$",
        'Tv': r"$T_{\text{v}}$"
    },
    'format': {
        'Y_O2': '%.2f',
        'Y_O': '%.2f',
        'Th': None,
        'Tv': None
    },
    'cticks': {
        'Y_O2': True,
        'Y_O': True,
        'Th': False,
        'Tv': False
    }
}
path = os.path.abspath(os.getcwd() + '/../output/resolution/') + '/'

Read data

In [ ]:
data = read_dataset(models, scheme, nb_iters, nb_nodes)

Compute error

In [ ]:
def capture_shock(x, eps=1.e-2):
    idx = (np.absolute((x - x[0,0])/x[0,0]) > eps).argmax(axis=1)
    return idx# - idx[0]

def mape(y_pred, y_true):
    return 100 * np.absolute((y_true - y_pred) / y_true)

# def compute_err(y, shift, filter_size=2, safe_idx=5):
#     err = mape(y['surrogate'], y['thermolib'])
#     err_ = []
#     for i, err_i in enumerate(err):
#         idx = min(shift['thermolib'][i], shift['surrogate'][i])
#         err_.append(np.roll(err_i, -idx))
#     err = np.vstack(err_)
#     return err

def compute_err(y, shift, filter_size=1, safe_idx=2):
    # Error
    err = mape(y['surrogate'], y['thermolib'])
    # Shock reference frame
    err_ = []
    for i, err_i in enumerate(err):
        idx = max(shift['thermolib'][i], shift['surrogate'][i]) + safe_idx
        err_i[:idx] = 0.
        err_.append(err_i)
    err = np.vstack(err_)
    # Clipping
    # idx = shift['thermolib'][0]
    # err = np.clip(err, 0., 5)
    # Window avarage
    err = ndimage.uniform_filter(err, size=filter_size)
    err = np.vstack([
        np.roll(e, -int(safe_idx/2)) for i, e in enumerate(err)
    ])
    return err

# def compute_err(y, shift, filter_size=4, safe_idx=4):
#     err = mape(y['surrogate'], y['thermolib'])
#     err_ = []
#     for i, err_i in enumerate(err):
#         idx = max(shift['thermolib'][i], shift['surrogate'][i]) + safe_idx
#         err_i[:idx] = 0.
#         err_.append(err_i)
#     err = np.vstack(err_)
#     err = ndimage.uniform_filter(err, size=filter_size)
#     return err

# def compute_err(y, shift, filter_size=2):
#     err = mape(y['surrogate'], y['thermolib'])
#     err = np.vstack([
#         np.roll(e, shift['thermolib'][i]) for i, e in enumerate(err)
#     ])
#     for k, y_k in y.items():
#         y[k] = np.vstack([
#             np.roll(y_ki, -shift[k][i]) for i, y_ki in enumerate(y_k)
#         ])
#     err = mape(y['surrogate'], y['thermolib'])
#     err = ndimage.uniform_filter(err, size=filter_size)
#     err = np.vstack([
#         np.roll(e, shift['thermolib'][i]) for i, e in enumerate(err)
#     ])
#     return err

In [ ]:
# [capture_shock(data[model]['u']) for model in models]

In [ ]:
# shock_pos = [capture_shock(data[model]['u']) for model in models]
# shift_idx = np.diff(shock_pos, axis=0).squeeze()
# err = {k: mape(data['thermolib'][k], data['surrogate'][k], shift_idx) for k in qoi}

In [ ]:
shock_pos = {model: capture_shock(data[model]['u']) for model in models}
err = {
    k: compute_err(
        y={model: data[model][k] for model in models},
        shift=shock_pos
    ) for k in qoi['names']
}

Plot variables and error

In [ ]:
t = data['thermolib']['t']
x = data['thermolib']['x']
# for k in qoi:
    # for model in models:
    #     time_space_2d(
    #         t,
    #         x,
    #         data[model][k],
    #         tlim=[0., 1.e-5],
    #         xlim=[0.004, 0.009],
    #         xshift=-0.005,
    #         show=True,
    #         cbar_kwargs={
    #             'label': k,
    #             'format': '%.1e'
    #         }
    #     )
    # pltt.time_space_2d(
    #     t,
    #     x,
    #     err[k],
    #     tlim=[0., 1.e-5],
    #     xlim=[0.004, 0.009],
    #     xshift=-0.005,
    #     # xlim=[0.00505, 0.009],
    #     # xshift=-0.00505,
    #     show=True,
    #     cbar_kwargs={
    #         'label': 'Relative error [%]',
    #         'format': '%.1f'
    #     }
    # )

### Plot space-time diagram

In [ ]:
t = data['thermolib']['t']
x = data['thermolib']['x']
saving_path = path + '/space.time/'
os.makedirs(saving_path, exist_ok=True)

In [ ]:
for k in qoi['names']:
    for model in models:
        pltt.time_space_2d(
            t,
            x,
            y=data[model][k],
            tlim=[0., 1.e-5],
            xlim=[0.004, 0.009],
            clbl=qoi['labels'][k],
            cfmt=qoi['format'][k],
            compute_cticks=qoi['cticks'][k],
            interp='none',
            figname=f'{model}.{k}',
            path=saving_path
        )

### Plot final profiles

In [ ]:
# Global parameters
x = data['thermolib']['x']
xlim = [0.004, 0.008]
shift = shock_pos['thermolib'][-1] - shock_pos['surrogate'][-1]
saving_path = path + '/profiles/'
os.makedirs(saving_path, exist_ok=True)

Temperatures

In [ ]:
keys = ('Th', 'Tv')
labels = [r"$x$ [m]", r"$T$ [K]"]

# Collect data
y = np.vstack([data['thermolib'][k][-1] for k in keys]).T
y_pred = np.vstack([data['surrogate'][k][-1] for k in keys]).T
y_pred = np.roll(y_pred, shift, axis=0)

# Plot profiles
pltt.test(
    x,
    y,
    y_pred,
    var_names=[qoi['labels'][k] for k in keys],
    fig_name='temp',
    labels=labels,
    xlim=xlim,
    path=saving_path
)

Mass fractions

In [ ]:
keys = ('Y_O2', 'Y_O')
labels = [r"$x$ [m]", r"$Y$"]

# Collect data
y = np.vstack([data['thermolib'][k][-1] for k in keys]).T
y_pred = np.vstack([data['surrogate'][k][-1] for k in keys]).T
y_pred = np.roll(y_pred, shift, axis=0)

# Plot profiles
pltt.test(
    x,
    y,
    y_pred,
    var_names=[qoi['labels'][k] for k in keys],
    fig_name='mass',
    labels=labels,
    xlim=xlim,
    path=saving_path
)